# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sandhyamanne994-create/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Plain-Words Data Contract Answers (1-3: Grain, Tables, Time Window)
1. **What one row means (Grain):** One row represents **one pseudonymized content item (page)** for a specific client over a defined observation window (e.g., trailing 90-day search visibility and user analytics metrics per content item, or daily performance aggregated per page).
2. **Which table(s) will be used:** `fact_content_daily_performance` (partitioned by month, e.g. `month=2026-03`) joined with `dim_content` and `dim_clients` from Hugging Face dataset `FlyRank/internship-warehouse`. For starter playground experiments, we use `data/raw/content_refresh_anonymized.csv`.
3. **Which time window:** Mid-panel month partition (`month=2026-03`, spanning March 1 to March 31, 2026, or trailing 90 days prior to decision moment). The final month (`2026-06`) is strictly sealed as a test window to prevent peeking into the future.

In [1]:
import pandas as pd
import numpy as np
import duckdb

# Connect DuckDB and load the slice
con = duckdb.connect()
con.execute("CREATE TABLE content AS SELECT * FROM read_csv_auto('data/raw/content_refresh_anonymized.csv')")

# Load active mature pages slice (impressions > 0, age >= 90 days)
df_raw = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df_slice = df_raw[(df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)].copy()

print(f"Raw dataset: {len(df_raw):,} pages | {df_raw['client_id'].nunique()} clients")
print(f"Active mature slice (impressions_90d > 0 & age >= 90d): {len(df_slice):,} pages")
print(f"Time window span: Trailing 90-day observation window (content age range: {df_slice['content_age_days'].min()} to {df_slice['content_age_days'].max()} days)")

Raw dataset: 30,000 pages | 32 clients
Active mature slice (impressions_90d > 0 & age >= 90d): 30,000 pages
Time window span: Trailing 90-day observation window (content age range: 90 to 564 days)



## 2. Fields: feature / label / context / excluded

### Plain-Words Data Contract Answers (4-5: Target/Proxy & Excluded Fields)
4. **What you'd predict or rank (Label or proxy):** `is_declining_label` — a binary observed outcome proxy (1 = organic search trend direction is 'down', 0 = stable/growing) computed from trailing organic search traffic trends. Evaluated via Precision@K (e.g. Precision@20, Precision@50) for ranking review candidates.
5. **One thing deliberately excluded & why:** `trend_direction` and `trend_pct` — deliberately excluded from feature inputs because `is_declining_label` is derived directly from `trend_direction` which is calculated from `trend_pct`. Including them as features creates **catastrophic target leakage** (feeding the label back into the model). We also exclude identifier hashes (`content_id`, `client_id`, `url_hash_id`, `keyword_hash_id`) from features, keeping them strictly as context for grouping and joining.

### Complete Field Classification Table

| Field | Bucket | Role & Description | Available When? |
|---|---|---|---|
| `impressions_90d` | Feature | Total search impressions over trailing 90d | Knowable prior to decision moment |
| `avg_position` | Feature | Average GSC ranking position | Knowable prior to decision moment |
| `days_since_last_update` | Feature | Days since content was last updated in CMS | Knowable prior to decision moment |
| `word_count` | Feature | Published article word count | Knowable prior to decision moment |
| `engagement_rate` | Feature | Trailing 90-day user engagement rate (%) | Knowable prior to decision moment |
| `is_declining_label` | Target / Proxy | Binary flag (1 = trend is 'down') | Target label to predict |
| `content_id`, `client_id` | Context | Pseudonymized identifiers | For grouping, splits & joins only |
| `trend_direction`, `trend_pct` | Excluded | Trend direction and percentage | EXCLUDED: Direct source of target label (leakage trap) |

In [2]:
# Create target column and define field classification buckets
df_raw['is_declining_label'] = df_raw['trend_direction'].astype(str).str.lower().eq('down').astype(int)
df_slice['is_declining_label'] = df_slice['trend_direction'].astype(str).str.lower().eq('down').astype(int)

context_cols = ['content_id', 'client_id']
feature_cols = ['impressions_90d', 'avg_position', 'days_since_last_update', 'word_count', 'engagement_rate']
target_col = 'is_declining_label'
excluded_cols = ['trend_direction', 'trend_pct']

print(f"Target distribution in slice:")
print(df_slice[target_col].value_counts(normalize=True).rename({0: 'Stable/Growing (0)', 1: 'Declining (1)'}).map('{:.1%}'.format))
print(f"\nField classification summary:")
print(f"  Features ({len(feature_cols)}): {feature_cols}")
print(f"  Target (1): {target_col}")
print(f"  Context ({len(context_cols)}): {context_cols}")
print(f"  Excluded ({len(excluded_cols)}): {excluded_cols}")

Target distribution in slice:
is_declining_label
Declining (1)         54.2%
Stable/Growing (0)    45.8%
Name: proportion, dtype: str

Field classification summary:
  Features (5): ['impressions_90d', 'avg_position', 'days_since_last_update', 'word_count', 'engagement_rate']
  Target (1): is_declining_label
  Context (2): ['content_id', 'client_id']
  Excluded (2): ['trend_direction', 'trend_pct']



## 3. Verify it with queries (grain, counts, missing values, windows)

### Verification Plan & 3 Core Queries
Every contract claim must be verified with executable SQL queries on real warehouse data:
1. **Query 1 (Grain verification):** Probe for duplicate `content_id` rows — expecting 0 duplicate rows (`HAVING COUNT(*) > 1` returning empty).
2. **Query 2 (Slice row count & metric span):** Measure total row count, distinct clients, and span of content age and search impressions on the active mature slice.
3. **Query 3 (Availability check with IS TRUE):** Check metric availability using `IS TRUE` logic (verifying high-demand tracking availability `impressions_90d >= 500 IS TRUE` and tracking coverage).

### 5-Feature Frame & The Deliberate Leakage Trap
- Build a 5-feature frame where every feature includes a one-line availability justification: *"knowable at the decision moment because..."*
- **The Trap:** Deliberately add `trend_pct` as a leaky feature, fit a decision tree to observe the score jump to 1.000, then DELETE it to report the honest score.

In [3]:
import duckdb
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
con.execute("CREATE TABLE content AS SELECT * FROM read_csv_auto('data/raw/content_refresh_anonymized.csv')")

print("=== QUERY 1: GRAIN VERIFICATION (One row per content_id) ===")
q1 = con.sql("""
    SELECT content_id, COUNT(*) AS cnt
    FROM content
    GROUP BY content_id
    HAVING cnt > 1
    LIMIT 5
""").df()
print(f"Duplicate rows returned: {len(q1)} (0 rows returned confirms one row per content item grain holds!)\n")

print("=== QUERY 2: SLICE ROW COUNT AND METRIC SPAN ===")
q2 = con.sql("""
    SELECT 
        COUNT(*) AS total_slice_rows,
        COUNT(DISTINCT client_id) AS distinct_clients,
        MIN(content_age_days) AS min_age_days,
        MAX(content_age_days) AS max_age_days,
        MIN(impressions_90d) AS min_impressions,
        MAX(impressions_90d) AS max_impressions
    FROM content
    WHERE impressions_90d > 0 AND content_age_days >= 90
""").df()
print(q2.to_string(index=False))
print()

print("=== QUERY 3: AVAILABILITY CHECK (Filter with IS TRUE) ===")
q3 = con.sql("""
    SELECT 
        COUNT(*) AS total_active_mature_rows,
        SUM(CASE WHEN (impressions_90d >= 500) IS TRUE THEN 1 ELSE 0 END) AS high_demand_rows,
        ROUND(AVG(CASE WHEN (impressions_90d >= 500) IS TRUE THEN 1.0 ELSE 0.0 END), 4) AS high_demand_availability_share
    FROM content
    WHERE impressions_90d > 0 AND content_age_days >= 90
""").df()
print(q3.to_string(index=False))
print()

print("=== 5-FEATURE FRAME (with 'knowable at decision moment' lines) ===")
feature_frame = df_slice[['content_id', 'impressions_90d', 'avg_position', 'days_since_last_update', 'word_count', 'engagement_rate']].head(5).copy()
print(feature_frame.to_string(index=False))
print("\nFeature availability justification:")
print("1. impressions_90d: knowable at the decision moment because total search impressions are logged during the trailing 90-day window prior to review.")
print("2. avg_position: knowable at the decision moment because average search rank is continuously tracked by Google Search Console before review.")
print("3. days_since_last_update: knowable at the decision moment because article publish and edit timestamps are recorded in the CMS prior to review.")
print("4. word_count: knowable at the decision moment because published document word count is extracted directly from the article body before review.")
print("5. engagement_rate: knowable at the decision moment because user interaction events are measured during the trailing observation window prior to review.")
print()

print("=== THE TRAP: DELIBERATE LEAKAGE EXPERIMENT ===")
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

# Prepare features and target
y = df_slice['is_declining_label'].values
X_honest = df_slice[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# 1. Add leaky feature on purpose
X_leaky = X_honest.copy()
X_leaky['leaky_trend_pct'] = df_slice['trend_pct'].values

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
scores_leaky = tree_leaky.predict_proba(X_leaky)[:, 1]
auc_leaky = roc_auc_score(y, scores_leaky)
p50_leaky = precision_at_k(scores_leaky, y, 50)
print(f"[TRAP ACTIVE] With Leaky Feature ('trend_pct'):  ROC-AUC = {auc_leaky:.4f} | Precision@50 = {p50_leaky:.4f} (Artificially Perfect!)")

# 2. Delete leaky feature and compute honest score
tree_honest = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42).fit(X_honest, y)
scores_honest = tree_honest.predict_proba(X_honest)[:, 1]
auc_honest = roc_auc_score(y, scores_honest)
p50_honest = precision_at_k(scores_honest, y, 50)
print(f"[TRAP REMOVED] Honest Feature Matrix (No Leakage): ROC-AUC = {auc_honest:.4f} | Precision@50 = {p50_honest:.4f} (Honest Baseline!)")

=== QUERY 1: GRAIN VERIFICATION (One row per content_id) ===
Duplicate rows returned: 0 (0 rows returned confirms one row per content item grain holds!)

=== QUERY 2: SLICE ROW COUNT AND METRIC SPAN ===
 total_slice_rows  distinct_clients  min_age_days  max_age_days  min_impressions  max_impressions
            30000                32            90           564                1           517715

=== QUERY 3: AVAILABILITY CHECK (Filter with IS TRUE) ===
 total_active_mature_rows  high_demand_rows  high_demand_availability_share
                    30000           16726.0                          0.5575

=== 5-FEATURE FRAME (with 'knowable at decision moment' lines) ===
          content_id  impressions_90d  avg_position  days_since_last_update  word_count  engagement_rate
content_304f48230142             3803          10.6                      20      3221.0             5.88
content_a1fb4e703a9e            15320          20.3                      25      2481.0             0.00
content

## 4. Data limits

### Named Data Limitation
**Unbalanced Panel History & Tracking Coverage Gaps:**
1. **Unbalanced Panel History:** Client tracking start dates vary across the panel dataset (`gsc_data_start`). Newer clients have less historical depth, so earlier history for newly onboarded clients is absent rather than zero traffic.
2. **GA4 Tracking Gaps & Zero-Fills:** Historical rows before a client's `ga4_data_start` have GA4 engagement columns zero-filled with `ga4_data_available = FALSE`. Zero values in engagement metrics must be filtered via `ga4_data_available IS TRUE` rather than treated as zero user engagement.
3. **Observational Bounds:** This dataset provides observational search measurements only; it cannot prove causal recovery from content edits without a controlled experiment.

In [4]:
# Quantify data limitations on missingness and zero-fills
print("Data Limitation Audit:")
missing_stats = pd.DataFrame({
    'Column': ['avg_position', 'word_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'],
    'Missing_or_Zero_Count': [
        (df_slice['avg_position'] == 0).sum(),
        (df_slice['word_count'] == 0).sum(),
        df_slice['engagement_rate'].isna().sum(),
        df_slice['scroll_rate'].isna().sum(),
        df_slice['ai_traffic_pct'].isna().sum()
    ]
})
missing_stats['Percentage'] = (missing_stats['Missing_or_Zero_Count'] / len(df_slice)).map('{:.2%}'.format)
print(missing_stats.to_string(index=False))
print(f"\nKey takeaway: avg_position = 0 represents 'no search position data', not rank 0.")

Data Limitation Audit:
         Column  Missing_or_Zero_Count Percentage
   avg_position                   1205      4.02%
     word_count                      0      0.00%
engagement_rate                      0      0.00%
    scroll_rate                    125      0.42%
 ai_traffic_pct                      0      0.00%

Key takeaway: avg_position = 0 represents 'no search position data', not rank 0.



## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers provided (grain, tables, windows, label, excluded)
- [x] Exactly three verification queries executed with outputs visible (availability checked with IS TRUE)
- [x] Five-feature frame built with an 'available when?' line per feature
- [x] Deliberate-leak experiment shown and removed (leakage trap demonstrated)
- [x] One named limitation of the slice clearly documented
- [x] Committed to my repo under `work/notebooks/w03_data_contract.ipynb` — then submit repo URL on card. Done.